# TestSafebound

## Build

### Init

In [1]:
import pandas as pd
import sys
import os

# ==================== Configuration Area ====================
# Data directory config: can specify different data directories for IMDB and Stats
# Default uses Data directory under SafeBound root
# If you need custom data directories, modify the paths below
default_root = os.path.abspath('../methods/SafeBound/')
if not default_root.endswith('/'):
    default_root += '/'

# IMDB data directory (points to directory containing IMDB CSV files, e.g. /path/to/Data/IMDB/)
imdb_data_directory = default_root + "Data/IMDB/"

# Stats data directory (points to directory containing Stats CSV files, e.g. /path/to/Data/Stats/)
stats_data_directory = default_root + "Data/Stats/"

# Example: to use custom paths, uncomment and modify the lines below:
# imdb_data_directory = "/path/to/your/imdb/data/directory/"
# stats_data_directory = "/path/to/your/stats/data/directory/"
imdb_data_directory = os.path.abspath("../methods/SafeBound/Data/IMDB")
stats_data_directory = os.path.abspath("../methods/SafeBound/Data/Stats")
# ================================================

# Get SafeBound root directory (access methods/SafeBound/ from experiment/ directory)
rootFileDirectory = os.path.abspath('../methods/SafeBound/')
if not rootFileDirectory.endswith('/'):
    rootFileDirectory += '/'

# Add Source directory first (must be first so ExperimentUtils can be imported as a package)
# Note: This must be done before any possible import of experiment/ExperimentUtils.py
source_path = rootFileDirectory + 'Source'
experiment_utils_path = rootFileDirectory + 'Source/ExperimentUtils'

# If ExperimentUtils has been imported as a file module, clear it first (avoid naming conflicts)
if 'ExperimentUtils' in sys.modules:
    # Check if it's a file module (not a package)
    module = sys.modules['ExperimentUtils']
    if hasattr(module, '__file__') and 'experiment' in module.__file__.lower():
        # This is experiment/ExperimentUtils.py file, needs to be removed
        del sys.modules['ExperimentUtils']

# Add SafeBound's Source directory to the front of sys.path (highest priority)
if source_path not in sys.path:
    sys.path.insert(0, source_path)
if experiment_utils_path not in sys.path:
    sys.path.insert(1, experiment_utils_path)

# Before importing BuildUtils, set LoadUtils data directory first
# Since ExperimentUtils directory is already in sys.path, can import LoadUtils directly
import LoadUtils
LoadUtils.imdb_data_directory = imdb_data_directory
LoadUtils.stats_data_directory = stats_data_directory

# BuildUtils is also in the ExperimentUtils directory, can be imported directly
# (BuildUtils.py has already added Source and Source/ExperimentUtils to sys.path internally)
from BuildUtils import *

# Output to experiment/checkpoint/SafeBound directory
checkpoint_dir = os.path.abspath('./checkpoint/SafeBound')
os.makedirs(checkpoint_dir, exist_ok=True)

def build_safebound_benchmark(benchmark):
    """
    Build SafeBound statistics object

    Args:
        benchmark: benchmark name ('Stats', 'JOBLight', 'JOBLightRanges', 'JOBM')

    Returns:
        tuple: (build time, statistics object size)
    """
    # Directly use the third parameter configuration (index 2)
    parameters = {
        'relativeErrorPerSegment': 0.02,
        'numHistogramBuckets': 32,
        'numEqualityOutliers': 512,
        'numCDFGroups': 16,
        'trackNulls': False,
        'trackTriGrams': False,
        'numCores': 18,
        'groupingMethod': "CompleteClustering",
        'modelCDF': True,
        'verbose': False
    }

    # Special handling for JOBM
    if benchmark == 'JOBM':
        parameters['numEqualityOutliers'] = 5 * parameters['numEqualityOutliers']
        parameters['trackTriGrams'] = True
        parameters['trackNulls'] = True
        parameters['numCores'] = 6
        parameters['verbose'] = True

    outputFile = os.path.join(checkpoint_dir, f"SafeBound_3_{benchmark}.pkl")

    print(f"\n{'='*60}")
    print(f"Processing Benchmark: {benchmark}")
    print(f"Using parameter configuration:")
    for key, value in parameters.items():
        print(f"  {key}: {value}")
    print(f"Output file: {outputFile}")

    # Building statistics object
    time, size = build_stats_object(
        method='SafeBound',
        benchmark=benchmark,
        parameters=parameters,
        outputFile=outputFile
    )

    print(f"Build complete!")
    print(f"Build time: {time:.2f} seconds")
    print(f"Statistics object size: {size} bytes")

    return time, size

print(f"IMDB data directory configured as: {imdb_data_directory}")
print(f"Stats data directory configured as: {stats_data_directory}")
print("Configuration complete, use build_safebound_benchmark() to build each benchmark")

IMDB data directory configured as: /home/liwei/starce-final/StarCE/methods/SafeBound/Data/IMDB
Stats data directory configured as: /home/liwei/starce-final/StarCE/methods/SafeBound/Data/Stats
Configuration complete, use build_safebound_benchmark() to build each benchmark


### Stats Benchmark

In [2]:
stats_time, stats_size = build_safebound_benchmark('Stats')
stats_time, stats_size


Processing Benchmark: Stats
Using parameter configuration:
  relativeErrorPerSegment: 0.02
  numHistogramBuckets: 32
  numEqualityOutliers: 512
  numCDFGroups: 16
  trackNulls: False
  trackTriGrams: False
  numCores: 18
  groupingMethod: CompleteClustering
  modelCDF: True
  verbose: False
Output file: /home/liwei/starce-final/StarCE/experiment/checkpoint/SafeBound/SafeBound_3_Stats.pkl


/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/

Table: BADGES
Filter Col: USERS.DOWNVOTES
Interval Footprint: 5304
Range Footprint: 4176.0
Equality Bloom Filter Footprint: 4800.0
Equality Outlier Footprint: 1128.0
Equality Max Footprint: 48.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: USERS.UPVOTES
Interval Footprint: 5440
Range Footprint: 4128.0
Equality Bloom Filter Footprint: 4800.0
Equality Outlier Footprint: 1368.0
Equality Max Footprint: 48.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: USERS.VIEWS
Interval Footprint: 5440
Range Footprint: 3552.0
Equality Bloom Filter Footprint: 4800.0
Equality Outlier Footprint: 1224.0
Equality Max Footprint: 48.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: USERS.CREATIONDATE
Interval Footprint: 5440
Range Footprint: 4536.0
Equality Bloom Filter Footprint: 4912.0
Equality Outlier Footprint: 768.0
Equality Max Footprint: 48.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: BADGES.DATE
Interval Footprint: 5440
Range Footprint: 4440.0
Equality Bloom Filter Footprint: 5154

(21.998126, 1268042.0)

### JOBLight Benchmark

In [3]:
joblight_time, joblight_size = build_safebound_benchmark('JOBLight')
joblight_time, joblight_size


Processing Benchmark: JOBLight
Using parameter configuration:
  relativeErrorPerSegment: 0.02
  numHistogramBuckets: 32
  numEqualityOutliers: 512
  numCDFGroups: 16
  trackNulls: False
  trackTriGrams: False
  numCores: 18
  groupingMethod: CompleteClustering
  modelCDF: True
  verbose: False
Output file: /home/liwei/starce-final/StarCE/experiment/checkpoint/SafeBound/SafeBound_3_JOBLight.pkl


/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/

Table: CAST_INFO
Filter Col: CAST_INFO.ROLE_ID
Interval Footprint: 1772
Range Footprint: 3960.0
Equality Bloom Filter Footprint: 3300.0
Equality Outlier Footprint: 1392.0
Equality Max Footprint: 24.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: CAST_INFO.NR_ORDER
Interval Footprint: 5440
Range Footprint: 2952.0
Equality Bloom Filter Footprint: 4852.0
Equality Outlier Footprint: 600.0
Equality Max Footprint: 24.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: TITLE.EPISODE_NR
Interval Footprint: 5428
Range Footprint: 3480.0
Equality Bloom Filter Footprint: 4902.0
Equality Outlier Footprint: 1752.0
Equality Max Footprint: 24.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: TITLE.SEASON_NR
Interval Footprint: 5428
Range Footprint: 3408.0
Equality Bloom Filter Footprint: 4800.0
Equality Outlier Footprint: 1152.0
Equality Max Footprint: 24.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: TITLE.KIND_ID
Interval Footprint: 1092
Range Footprint: 2688.0
Equality Bloom Filter F

(242.268736, 362156.0)

### JOBLightRanges Benchmark

In [4]:
joblightranges_time, joblightranges_size = build_safebound_benchmark('JOBLightRanges')
joblightranges_time, joblightranges_size


Processing Benchmark: JOBLightRanges
Using parameter configuration:
  relativeErrorPerSegment: 0.02
  numHistogramBuckets: 32
  numEqualityOutliers: 512
  numCDFGroups: 16
  trackNulls: False
  trackTriGrams: False
  numCores: 18
  groupingMethod: CompleteClustering
  modelCDF: True
  verbose: False
Output file: /home/liwei/starce-final/StarCE/experiment/checkpoint/SafeBound/SafeBound_3_JOBLightRanges.pkl


/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/

Table: CAST_INFO
Filter Col: CAST_INFO.ROLE_ID
Interval Footprint: 1772
Range Footprint: 3960.0
Equality Bloom Filter Footprint: 3300.0
Equality Outlier Footprint: 1392.0
Equality Max Footprint: 24.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: CAST_INFO.NR_ORDER
Interval Footprint: 5440
Range Footprint: 2952.0
Equality Bloom Filter Footprint: 4852.0
Equality Outlier Footprint: 600.0
Equality Max Footprint: 24.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: TITLE.EPISODE_NR
Interval Footprint: 5428
Range Footprint: 3480.0
Equality Bloom Filter Footprint: 4902.0
Equality Outlier Footprint: 1752.0
Equality Max Footprint: 24.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: TITLE.SEASON_NR
Interval Footprint: 5428
Range Footprint: 3408.0
Equality Bloom Filter Footprint: 4800.0
Equality Outlier Footprint: 1152.0
Equality Max Footprint: 24.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: TITLE.PRODUCTION_YEAR
Interval Footprint: 5432
Range Footprint: 4416.0
Equality Bloom 

(391.001837, 608152.0)

### JOBM Benchmark

In [5]:
jobm_time, jobm_size = build_safebound_benchmark('JOBM')
jobm_time, jobm_size


Processing Benchmark: JOBM
Using parameter configuration:
  relativeErrorPerSegment: 0.02
  numHistogramBuckets: 32
  numEqualityOutliers: 2560
  numCDFGroups: 16
  trackNulls: True
  trackTriGrams: True
  numCores: 6
  groupingMethod: CompleteClustering
  modelCDF: True
  verbose: True
Output file: /home/liwei/starce-final/StarCE/experiment/checkpoint/SafeBound/SafeBound_3_JOBM.pkl
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Tabl

/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)


Building Stats: TITLE.PRODUCTION_YEAR


/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)


Building Stats: TITLE.TITLE
Building Stats: TITLE.EPISODE_NR
Building Stats: TITLE.PRODUCTION_YEAR
Building Stats: TITLE.TITLE


...

/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)


Building Stats: INFO_TYPE.INFO


/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)


Clustering Function Approximations For Most Common TriGrams: MOVIE_INFO.NOTE 1436962 Memory Usage: 1.66955008
Outlier Group Sets: MOVIE_INFO.NOTE 16
Handling Remainder Rows: MOVIE_INFO.NOTE 1436962 Memory Usage: 1.721335808
Finished Building Stats: MOVIE_INFO.NOTE
Approximating null distributions: TITLE.TITLE
Approximating not null distributions: TITLE.TITLE
Approximating range distributions: TITLE.TITLE Memory Usage: 1.850273792
Approximating equality distributions: TITLE.TITLE Memory Usage: 1.854681088
Detecting Most Common TriGrams: TITLE.TITLE 14835717 Memory Usage: 1.823141888
Creating Function Approximations For Most Common TriGrams: TITLE.TITLE 14835717 Memory Usage: 3.921637376
Clustering Function Approximations For Most Common TriGrams: TITLE.TITLE 14835717 Memory Usage: 3.46896384
Outlier Group Sets: TITLE.TITLE 16
Handling Remainder Rows: TITLE.TITLE 14835717 Memory Usage: 3.4714624
Finished Building Stats: TITLE.TITLE
Approximating null distributions: TITLE.EPISODE_NR
Appro

(2531.138521, 3581644.0)

### StatsJoin Benchmark

In [6]:
# StatsJoin reuses Stats statistics object (same database, same schema, just depredicated version)
statsjoin_time, statsjoin_size = build_safebound_benchmark('Stats')
statsjoin_time, statsjoin_size


Processing Benchmark: Stats
Using parameter configuration:
  relativeErrorPerSegment: 0.02
  numHistogramBuckets: 32
  numEqualityOutliers: 512
  numCDFGroups: 16
  trackNulls: False
  trackTriGrams: False
  numCores: 18
  groupingMethod: CompleteClustering
  modelCDF: True
  verbose: False
Output file: /home/liwei/starce-final/StarCE/experiment/checkpoint/SafeBound/SafeBound_3_Stats.pkl


/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/

Table: BADGES
Filter Col: USERS.VIEWS
Interval Footprint: 5440
Range Footprint: 3552.0
Equality Bloom Filter Footprint: 4800.0
Equality Outlier Footprint: 1224.0
Equality Max Footprint: 48.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: USERS.UPVOTES
Interval Footprint: 5440
Range Footprint: 4128.0
Equality Bloom Filter Footprint: 4800.0
Equality Outlier Footprint: 1368.0
Equality Max Footprint: 48.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: USERS.REPUTATION
Interval Footprint: 5428
Range Footprint: 3504.0
Equality Bloom Filter Footprint: 4812.0
Equality Outlier Footprint: 1248.0
Equality Max Footprint: 48.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: USERS.DOWNVOTES
Interval Footprint: 5304
Range Footprint: 4176.0
Equality Bloom Filter Footprint: 4800.0
Equality Outlier Footprint: 1128.0
Equality Max Footprint: 48.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: BADGES.DATE
Interval Footprint: 5440
Range Footprint: 4440.0
Equality Bloom Filter Footprint: 5154.

(20.81978, 1268042.0)

## Evaluate

### Init

In [7]:
import pickle
import time
import json
import sys
import os
import re

# Add SafeBound Source directory to path
sys.path.append(rootFileDirectory + 'Source')
from SafeBoundUtils import *
from JoinGraphUtils import *
from SQLParser import *

def evaluate_safebound_benchmark(benchmark, sql_file_path, stat_object_path):
    """
    Evaluate SafeBound statistics object cardinality estimation on SQL queries

    Args:
        benchmark: benchmark name ('Stats', 'JOBLight', 'JOBLightRanges', 'JOBM')
        sql_file_path: SQL query file path
        stat_object_path: statistics object pickle file path

    Returns:
        dict: Dictionary containing evaluation results, including:
            - total_time: Pure estimation time (seconds), excluding SQL parsing
            - parse_time: SQL parsing time (seconds)
            - num_queries: Number of queries
            - results: List of estimation results for each query
            - output_file: Result output file path
    """
    # Load statistics object
    print(f"\n{'='*60}")
    print(f"Loading statistics object: {stat_object_path}")
    safeBound = pickle.load(open(stat_object_path, 'rb'))
    print(f"Statistics object size: {os.path.getsize(stat_object_path)} bytes")

    # Read SQL query file
    print(f"\nReading SQL query file: {sql_file_path}")
    with open(sql_file_path, 'r') as f:
        sqls = f.readlines()

    print(f"Number of queries: {len(sqls)}")

    # Pre-parsing all SQL queries (parsing not counted towards estimation time)
    print(f"\nPre-parsing {len(sqls)} SQL queries...")
    parse_start = time.time()
    parsed_entries = []  # each element is (line_id, sql, query) or None (parsing failed)
    parse_failures = 0
    for line_id, sql in enumerate(sqls, 1):
        if (line_id - 1) % max(1, len(sqls) // 10) == 0 and line_id > 1:
            print(f"  Pre-parsing progress: {line_id - 1}/{len(sqls)}")
        try:
            if benchmark == 'Stats':
                sql_clean = sql.strip()
                if not sql_clean:  # skip empty lines
                    parsed_entries.append(None)
                    continue
                query = sql_to_joingraph(sql_clean, keep_type_cast=False)
            else:
                result = SQLQueriesToJoinQueryGraphs(sql)
                if len(result) > 0:
                    query = result[0]
                else:
                    print(f"Warning: query {line_id} parsing failed")
                    parsed_entries.append(None)
                    parse_failures += 1
                    continue
            query.buildJoinGraph()
            parsed_entries.append((line_id, sql, query))
        except Exception as e:
            print(f"Warning: query {line_id} parsing exception: {str(e)}")
            parsed_entries.append(None)
            parse_failures += 1
    parse_time = time.time() - parse_start
    successful_parses = len([e for e in parsed_entries if e is not None])
    print(f"Pre-parsing complete: succeeded {successful_parses}, failed {parse_failures}, took {parse_time:.2f}s（not counted towards estimation time）")

    # Timing: only measure cardinality estimation time (SQL parsing and file I/O excluded)
    results = []
    start_time = time.time()

    # Save results to checkpoint directory (one estimate per line)
    output_file = os.path.join(checkpoint_dir, f"SafeBound_3_{benchmark}_evaluate_results.txt")

    with open(output_file, 'w') as f:
        latencies = []
        estimate_count = 0
        estimate_failures = 0
        for entry in parsed_entries:
            if entry is None:
                f.write('NaN\n')
                latencies.append(0.0)
                continue
            line_id, sql, query = entry
            try:
                # Execute cardinality estimation
                _t0 = time.time()
                bound = safeBound.functionalFrequencyBound(query)
                latencies.append(time.time() - _t0)
                f.write(str(bound) + '\n')
                results.append({
                    'query_id': line_id,
                    'sql': sql,
                    'estimate': bound
                })

                estimate_count += 1
                if estimate_count % max(1, successful_parses // 10) == 0 or estimate_count == 1:
                    print(f"Estimated {estimate_count}/{successful_parses} queries...")

            except Exception as e:
                print(f"Error: estimating query {line_id} failed: {str(e)}")
                f.write('NaN\n')
                latencies.append(0.0)
                estimate_failures += 1
                results.append({
                    'query_id': line_id,
                    'sql': sql,
                    'estimate': None,
                    'error': str(e)
                })

    # Save per-subquery estimation time
    bench_display = 'STATS' if benchmark == 'Stats' else benchmark
    time_file = os.path.join(checkpoint_dir, f"estimate_time_{bench_display}.txt")
    with open(time_file, 'w') as tf:
        for lat in latencies:
            tf.write(f"{lat}\n")

    total_time = time.time() - start_time

    print(f"\nEvaluation complete!")
    print(f"Pre-parsing took (not counted towards estimation time): {parse_time:.2f} seconds")
    print(f"Total estimation time: {total_time:.2f} seconds")
    print(f"Average per query: {total_time/len(results):.4f} seconds" if len(results) > 0 else "")
    print(f"Results saved to: {output_file}")
    print(f"Successfully estimated queries: {sum(1 for r in results if r.get('estimate') is not None)}/{successful_parses}")
    print(f"Parsing failures: {parse_failures}")
    print(f"Estimation failures: {estimate_failures}")

    # Return summary information
    return {
        'benchmark': benchmark,
        'total_time': total_time,
        'parse_time': parse_time,
        'num_queries': len(results),
        'successful_queries': sum(1 for r in results if r.get('estimate') is not None),
        'parse_failures': parse_failures,
        'estimate_failures': estimate_failures,
        'output_file': output_file,
        'time_file': time_file
    }

class RawSqlValue:
    def __init__(self, value):
        self.value = value

    def __str__(self):
        return self.value

    def __eq__(self, other):
        if isinstance(other, RawSqlValue):
            return self.value == other.value
        if isinstance(other, str):
            return self.value == other
        return NotImplemented

    def __lt__(self, other):
        if isinstance(other, RawSqlValue):
            return self.value < other.value
        if isinstance(other, str):
            return self.value < other
        return NotImplemented

    def __le__(self, other):
        if isinstance(other, RawSqlValue):
            return self.value <= other.value
        if isinstance(other, str):
            return self.value <= other
        return NotImplemented

    def __gt__(self, other):
        if isinstance(other, RawSqlValue):
            return self.value > other.value
        if isinstance(other, str):
            return self.value > other
        return NotImplemented

    def __ge__(self, other):
        if isinstance(other, RawSqlValue):
            return self.value >= other.value
        if isinstance(other, str):
            return self.value >= other
        return NotImplemented


def parse_sql_value(raw_value, keep_type_cast=True):
    raw_value = raw_value.strip()
    if raw_value.upper() == "NULL":
        return RawSqlValue("NULL")
    if raw_value.startswith("'"):
        if "::" in raw_value:
            if keep_type_cast:
                return RawSqlValue(raw_value)
            value_part = raw_value.split("::", 1)[0].strip()
            if value_part.endswith("'") and len(value_part) >= 2:
                return value_part[1:-1]
            return value_part
        if raw_value.endswith("'") and len(raw_value) >= 2:
            return raw_value[1:-1]
        return RawSqlValue(raw_value)
    if re.match(r"^-?\d+$", raw_value):
        return int(raw_value)
    if re.match(r"^-?\d+\.\d+$", raw_value):
        return float(raw_value)
    return RawSqlValue(raw_value)


def sql_to_joingraph(sql_query, keep_type_cast=True):
    """
    Parse SQL query and convert to JoinQueryGraph (for Stats benchmark)

    Args:
        sql_query: SQL query string
        keep_type_cast: whether to preserve type casts like ::timestamp

    Returns:
        JoinQueryGraph instance
    """
    import re
    query = JoinQueryGraph()

    # Parse FROM clause to get table aliases
    from_match = re.search(r'FROM\s+(.+?)(?:\s+WHERE|$)', sql_query, re.IGNORECASE)
    if not from_match:
        raise ValueError("Invalid SQL query: missing FROM clause")

    # Parse table aliases
    tables = [t.strip() for t in from_match.group(1).split(',')]
    for table in tables:
        if ' AS ' in table.upper():
            table_name, alias = re.split(r'\s+AS\s+', table, flags=re.IGNORECASE)
        else:
            table_name = alias = table.split()[-1]
        query.addAlias(table_name.strip(), alias.strip())

    # Parse WHERE clause to get join conditions and filter predicates
    where_match = re.search(r'WHERE\s+(.+?)(?:\s*;|$)', sql_query, re.IGNORECASE)
    if where_match:
        join_conditions = re.split(r'\s+AND\s+', where_match.group(1), flags=re.IGNORECASE)

        for condition in join_conditions:
            condition = condition.strip().rstrip(';')
            # Parse conditions like table1.column1=table2.column2
            match = re.match(r'(\w+)\.(\w+)\s*=\s*(\w+)\.(\w+)$', condition, flags=re.IGNORECASE)
            if match:
                table1, col1, table2, col2 = match.groups()
                query.addJoin(table1, col1, table2, col2)
                continue

            pred_match = re.match(r'(\w+)\.(\w+)\s*(=|<=|>=|<|>)\s*(.+)$', condition, flags=re.IGNORECASE)
            if pred_match:
                alias, col, op, raw_value = pred_match.groups()
                value = parse_sql_value(raw_value, keep_type_cast=keep_type_cast)
                query.addPredicate(alias, col, op, value)

    return query


def collect_predicate_examples(sqls, max_per_type=2):
    import re
    examples = {}
    for idx, sql in enumerate(sqls, 1):
        where_match = re.search(r'WHERE\s+(.+?)(?:\s*;|$)', sql, re.IGNORECASE)
        if not where_match:
            continue
        conditions = re.split(r'\s+AND\s+', where_match.group(1), flags=re.IGNORECASE)
        for condition in conditions:
            condition = condition.strip().rstrip(';')
            join_match = re.match(r'(\w+)\.(\w+)\s*=\s*(\w+)\.(\w+)$', condition, flags=re.IGNORECASE)
            if join_match:
                continue
            pred_match = re.match(r'(\w+)\.(\w+)\s*(=|<=|>=|<|>)\s*(.+)$', condition, flags=re.IGNORECASE)
            if not pred_match:
                continue
            _, _, op, raw_value = pred_match.groups()
            raw_value = raw_value.strip()
            value_type = 'timestamp' if '::' in raw_value else 'numeric' if re.match(r'^-?\d+(\.\d+)?$', raw_value) else 'other'
            key = f"{op}:{value_type}"
            if key not in examples:
                examples[key] = []
            if len(examples[key]) < max_per_type:
                examples[key].append(idx)
    return examples


def print_sql_roundtrip(sqls, indices):
    for idx in indices:
        if idx < 1 or idx > len(sqls):
            print(f"Index out of range: {idx}")
            continue
        sql = sqls[idx - 1].strip()
        print(f"\n--- SQL #{idx} original ---")
        print(sql)
        try:
            query = sql_to_joingraph(sql, keep_type_cast=True)
            query.buildJoinGraph()
            print("--- getSQLQuery ---")
            print(query.getSQLQuery())
        except Exception as e:
            print(f"parsing failed: {e}")

print("Evaluation function defined, use evaluate_safebound_benchmark() to evaluate each benchmark")

Evaluation function defined, use evaluate_safebound_benchmark() to evaluate each benchmark


### Stats Benchmark

In [8]:
# ==================== Configuration Area ====================
# Stats Benchmark subquery file path
# Please modify this path as needed
# stats_sql_file = os.path.join(rootFileDirectory, 'subqueries-stats.sql')
# If the file is not under the SafeBound directory, use an absolute path, e.g.:
stats_sql_file = os.path.abspath('../Benchmark/workloads/STATS-CEB/subquery/subquery.sql')
# ================================================

# Statistics object path (generated during Build phase)
stats_stat_object = os.path.join(checkpoint_dir, 'SafeBound_3_Stats.pkl')

# Check if file exists
if not os.path.exists(stats_stat_object):
    print(f"Error: statistics object file not found: {stats_stat_object}")
    print("Please run the Build phase for Stats Benchmark")
elif not os.path.exists(stats_sql_file):
    print(f"Error: SQL file not found: {stats_sql_file}")
    print("Please check and modify stats_sql_file  path")
else:
    # Execute evaluation
    stats_results = evaluate_safebound_benchmark(
        benchmark='Stats',
        sql_file_path=stats_sql_file,
        stat_object_path=stats_stat_object
    )
    stats_results


Loading statistics object: /home/liwei/starce-final/StarCE/experiment/checkpoint/SafeBound/SafeBound_3_Stats.pkl
Statistics object size: 1406833 bytes

Reading SQL query file: /home/liwei/starce-final/StarCE/Benchmark/workloads/STATS-CEB/subquery/subquery.sql
Number of queries: 2471

Pre-parsing 2471 SQL queries...
  Pre-parsing progress: 247/2471
  Pre-parsing progress: 494/2471
  Pre-parsing progress: 741/2471
  Pre-parsing progress: 988/2471
  Pre-parsing progress: 1235/2471
  Pre-parsing progress: 1482/2471
  Pre-parsing progress: 1729/2471
  Pre-parsing progress: 1976/2471
  Pre-parsing progress: 2223/2471
  Pre-parsing progress: 2470/2471
Pre-parsing complete: succeeded 2471, failed 0, took 0.40s（not counted towards estimation time）
Estimated 1/2471 queries...
Estimated 247/2471 queries...
Estimated 494/2471 queries...
Estimated 741/2471 queries...
Estimated 988/2471 queries...
Estimated 1235/2471 queries...
Estimated 1482/2471 queries...
Estimated 1729/2471 queries...
Estimated

### JOBLight Benchmark

In [9]:
# ==================== Configuration Area ====================
# JOBLight Benchmark subquery file path
# Please modify this path as needed
# If the file is not under the SafeBound directory, use an absolute path, e.g.:
joblight_sql_file = os.path.abspath('../Benchmark/workloads/JOBLight/subquery/subquery2.sql')
# ================================================

# Statistics object path (generated during Build phase)
joblight_stat_object = os.path.join(checkpoint_dir, 'SafeBound_3_JOBLight.pkl')

# Check if file exists
if not os.path.exists(joblight_stat_object):
    print(f"Error: statistics object file not found: {joblight_stat_object}")
    print("Please run the Build phase for JOBLight Benchmark")
elif not os.path.exists(joblight_sql_file):
    print(f"Error: SQL file not found: {joblight_sql_file}")
    print("Please check and modify joblight_sql_file  path")
else:
    # Execute evaluation
    joblight_results = evaluate_safebound_benchmark(
        benchmark='JOBLight',
        sql_file_path=joblight_sql_file,
        stat_object_path=joblight_stat_object
    )
    joblight_results


Loading statistics object: /home/liwei/starce-final/StarCE/experiment/checkpoint/SafeBound/SafeBound_3_JOBLight.pkl
Statistics object size: 378926 bytes

Reading SQL query file: /home/liwei/starce-final/StarCE/Benchmark/workloads/JOBLight/subquery/subquery2.sql
Number of queries: 451

Pre-parsing 451 SQL queries...
  Pre-parsing progress: 45/451
  Pre-parsing progress: 90/451
  Pre-parsing progress: 135/451
  Pre-parsing progress: 180/451
  Pre-parsing progress: 225/451
  Pre-parsing progress: 270/451
  Pre-parsing progress: 315/451
  Pre-parsing progress: 360/451
  Pre-parsing progress: 405/451
  Pre-parsing progress: 450/451
Pre-parsing complete: succeeded 451, failed 0, took 3.89s（not counted towards estimation time）
Estimated 1/451 queries...
Estimated 45/451 queries...
Estimated 90/451 queries...
Estimated 135/451 queries...
Estimated 180/451 queries...
Estimated 225/451 queries...
Estimated 270/451 queries...
Estimated 315/451 queries...
Estimated 360/451 queries...
Estimated 40

### JOBLightRanges Benchmark

In [10]:
# ==================== Configuration Area ====================
# JOBLightRanges Benchmark subquery file path
# Please modify this path as needed
# If the file is not under the SafeBound directory, use an absolute path, e.g.:
joblightranges_sql_file = os.path.abspath('../Benchmark/workloads/JOBLightRanges/subquery/subquery2.sql')
# ================================================

# Statistics object path (generated during Build phase)
joblightranges_stat_object = os.path.join(checkpoint_dir, 'SafeBound_3_JOBLightRanges.pkl')

# Check if file exists
if not os.path.exists(joblightranges_stat_object):
    print(f"Error: statistics object file not found: {joblightranges_stat_object}")
    print("Please run the Build phase for JOBLightRanges Benchmark")
elif not os.path.exists(joblightranges_sql_file):
    print(f"Error: SQL file not found: {joblightranges_sql_file}")
    print("Please check and modify joblightranges_sql_file  path")
else:
    # Execute evaluation
    joblightranges_results = evaluate_safebound_benchmark(
        benchmark='JOBLightRanges',
        sql_file_path=joblightranges_sql_file,
        stat_object_path=joblightranges_stat_object
    )
    joblightranges_results


Loading statistics object: /home/liwei/starce-final/StarCE/experiment/checkpoint/SafeBound/SafeBound_3_JOBLightRanges.pkl
Statistics object size: 628524 bytes

Reading SQL query file: /home/liwei/starce-final/StarCE/Benchmark/workloads/JOBLightRanges/subquery/subquery2.sql
Number of queries: 8292

Pre-parsing 8292 SQL queries...
  Pre-parsing progress: 829/8292
  Pre-parsing progress: 1658/8292
  Pre-parsing progress: 2487/8292
  Pre-parsing progress: 3316/8292
  Pre-parsing progress: 4145/8292
  Pre-parsing progress: 4974/8292
  Pre-parsing progress: 5803/8292
  Pre-parsing progress: 6632/8292
  Pre-parsing progress: 7461/8292
  Pre-parsing progress: 8290/8292
Pre-parsing complete: succeeded 8292, failed 0, took 85.29s（not counted towards estimation time）
Estimated 1/8292 queries...
Estimated 829/8292 queries...
Estimated 1658/8292 queries...
Estimated 2487/8292 queries...
Estimated 3316/8292 queries...
Estimated 4145/8292 queries...
Estimated 4974/8292 queries...
Estimated 5803/8292

### JOBM Benchmark

In [11]:
# ==================== Configuration Area ====================
# JOBM Benchmark subquery file path
# Please modify this path as needed
# jobm_sql_file = os.path.join(rootFileDirectory, 'subqueries.sql')
# If the file is not under the SafeBound directory, use an absolute path, e.g.:
jobm_sql_file = os.path.abspath('../Benchmark/workloads/JOBM/subquery/subquery2.sql')
# ================================================

# Statistics object path (generated during Build phase)
jobm_stat_object = os.path.join(checkpoint_dir, 'SafeBound_3_JOBM.pkl')

# Check if file exists
if not os.path.exists(jobm_stat_object):
    print(f"Error: statistics object file not found: {jobm_stat_object}")
    print("Please run the Build phase for JOBM Benchmark")
elif not os.path.exists(jobm_sql_file):
    print(f"Error: SQL file not found: {jobm_sql_file}")
    print("Please check and modify jobm_sql_file  path")
else:
    # Execute evaluation
    jobm_results = evaluate_safebound_benchmark(
        benchmark='JOBM',
        sql_file_path=jobm_sql_file,
        stat_object_path=jobm_stat_object
    )
    jobm_results


Loading statistics object: /home/liwei/starce-final/StarCE/experiment/checkpoint/SafeBound/SafeBound_3_JOBM.pkl
Statistics object size: 1274650 bytes

Reading SQL query file: /home/liwei/starce-final/StarCE/Benchmark/workloads/JOBM/subquery/subquery2.sql
Number of queries: 6424

Pre-parsing 6424 SQL queries...
  Pre-parsing progress: 642/6424
  Pre-parsing progress: 1284/6424
  Pre-parsing progress: 1926/6424
  Pre-parsing progress: 2568/6424
  Pre-parsing progress: 3210/6424
  Pre-parsing progress: 3852/6424
  Pre-parsing progress: 4494/6424
  Pre-parsing progress: 5136/6424
  Pre-parsing progress: 5778/6424
  Pre-parsing progress: 6420/6424
Pre-parsing complete: succeeded 6424, failed 0, took 121.20s（not counted towards estimation time）
Estimated 1/6424 queries...
Estimated 642/6424 queries...
Estimated 1284/6424 queries...
Estimated 1926/6424 queries...
Estimated 2568/6424 queries...
Estimated 3210/6424 queries...
Estimated 3852/6424 queries...
Estimated 4494/6424 queries...
Estima

### StatsJoin Benchmark

In [12]:
# ==================== Configuration Area ====================
# StatsJoin Benchmark subquery file path (reuses Stats statistics object SafeBound_3_Stats.pkl)
statsjoin_sql_file = os.path.abspath('../Benchmark/workloads/StatsJoin/subquery/subquery.sql')
# ================================================

# Statistics object path (generated during Build phase, reuses Stats statistics object)
statsjoin_stat_object = os.path.join(checkpoint_dir, 'SafeBound_3_Stats.pkl')

# Check if file exists
if not os.path.exists(statsjoin_stat_object):
    print(f"Error: statistics object file not found: {statsjoin_stat_object}")
    print("Please run the Build phase for Stats Benchmark")
elif not os.path.exists(statsjoin_sql_file):
    print(f"Error: SQL file not found: {statsjoin_sql_file}")
    print("Please check and modify statsjoin_sql_file  path")
else:
    # Execute evaluation
    statsjoin_results = evaluate_safebound_benchmark(
        benchmark='StatsJoin',
        sql_file_path=statsjoin_sql_file,
        stat_object_path=statsjoin_stat_object
    )
    statsjoin_results


Loading statistics object: /home/liwei/starce-final/StarCE/experiment/checkpoint/SafeBound/SafeBound_3_Stats.pkl
Statistics object size: 1406833 bytes

Reading SQL query file: /home/liwei/starce-final/StarCE/Benchmark/workloads/StatsJoin/subquery/subquery.sql
Number of queries: 226

Pre-parsing 226 SQL queries...
  Pre-parsing progress: 22/226
  Pre-parsing progress: 44/226
  Pre-parsing progress: 66/226
  Pre-parsing progress: 88/226
  Pre-parsing progress: 110/226
  Pre-parsing progress: 132/226
  Pre-parsing progress: 154/226
  Pre-parsing progress: 176/226
  Pre-parsing progress: 198/226
  Pre-parsing progress: 220/226
Pre-parsing complete: succeeded 0, failed 226, took 0.08s（not counted towards estimation time）

Evaluation complete!
Pre-parsing took (not counted towards estimation time): 0.08 seconds
Total estimation time: 0.00 seconds

Results saved to: /home/liwei/starce-final/StarCE/experiment/checkpoint/SafeBound/SafeBound_3_StatsJoin_evaluate_results.txt
Successfully estimat

In [13]:
# Output each benchmark's evaluation time and build time as a CSV file
import csv
import os

# Complete benchmark list
all_benchmarks = {
    'Stats': {'BuildTime': None, 'EvaluationTime': None, 'StatisticsSize': None, 'ParseTime': None},
    'JOBLight': {'BuildTime': None, 'EvaluationTime': None, 'StatisticsSize': None, 'ParseTime': None},
    'JOBLightRanges': {'BuildTime': None, 'EvaluationTime': None, 'StatisticsSize': None, 'ParseTime': None},
    'JOBM': {'BuildTime': None, 'EvaluationTime': None, 'StatisticsSize': None, 'ParseTime': None},
    'StatsJoin': {'BuildTime': None, 'EvaluationTime': None, 'StatisticsSize': None, 'ParseTime': None}
}

# Get evaluation time from variables (if already run)
if 'stats_results' in locals() and stats_results:
    if isinstance(stats_results, dict) and 'total_time' in stats_results:
        all_benchmarks['Stats']['EvaluationTime'] = stats_results['total_time']
        all_benchmarks['Stats']['ParseTime'] = stats_results.get('parse_time')

if 'joblight_results' in locals() and joblight_results:
    if isinstance(joblight_results, dict) and 'total_time' in joblight_results:
        all_benchmarks['JOBLight']['EvaluationTime'] = joblight_results['total_time']
        all_benchmarks['JOBLight']['ParseTime'] = joblight_results.get('parse_time')

if 'joblightranges_results' in locals() and joblightranges_results:
    if isinstance(joblightranges_results, dict) and 'total_time' in joblightranges_results:
        all_benchmarks['JOBLightRanges']['EvaluationTime'] = joblightranges_results['total_time']
        all_benchmarks['JOBLightRanges']['ParseTime'] = joblightranges_results.get('parse_time')

if 'jobm_results' in locals() and jobm_results:
    if isinstance(jobm_results, dict) and 'total_time' in jobm_results:
        all_benchmarks['JOBM']['EvaluationTime'] = jobm_results['total_time']
        all_benchmarks['JOBM']['ParseTime'] = jobm_results.get('parse_time')

if 'statsjoin_results' in locals() and statsjoin_results:
    if isinstance(statsjoin_results, dict) and 'total_time' in statsjoin_results:
        all_benchmarks['StatsJoin']['EvaluationTime'] = statsjoin_results['total_time']
        all_benchmarks['StatsJoin']['ParseTime'] = statsjoin_results.get('parse_time')

# Get build time from variables (if already run and saved to variables)
# build_safebound_benchmark returns (time, size), saved as stats_time, joblight_time etc.
build_time_vars = {
    'Stats': 'stats_time',
    'JOBLight': 'joblight_time',
    'JOBLightRanges': 'joblightranges_time',
    'JOBM': 'jobm_time',
    'StatsJoin': 'statsjoin_time'
}

for benchmark, var_name in build_time_vars.items():
    if var_name in locals():
        var_value = locals()[var_name]
        # If it's a number, use directly
        if isinstance(var_value, (int, float)):
            all_benchmarks[benchmark]['BuildTime'] = var_value

# Prioritize getting statistics object size from variables
# build_safebound_benchmark returns (time, size), saved as stats_size, joblight_size etc.
size_vars = {
    'Stats': 'stats_size',
    'JOBLight': 'joblight_size',
    'JOBLightRanges': 'joblightranges_size',
    'JOBM': 'jobm_size',
    'StatsJoin': 'statsjoin_size'
}

for benchmark, var_name in size_vars.items():
    if var_name in locals():
        var_value = locals()[var_name]
        if isinstance(var_value, (int, float)):
            all_benchmarks[benchmark]['StatisticsSize'] = var_value

# If variable does not exist, get size from statistics object file path
stat_object_paths = {
    benchmark: os.path.join(checkpoint_dir, f"SafeBound_3_{benchmark}.pkl")
    for benchmark in all_benchmarks.keys()
}

for benchmark, stat_path in stat_object_paths.items():
    if all_benchmarks[benchmark]['StatisticsSize'] is None and os.path.exists(stat_path):
        try:
            all_benchmarks[benchmark]['StatisticsSize'] = os.path.getsize(stat_path)
        except OSError:
            all_benchmarks[benchmark]['StatisticsSize'] = None

print("Collecting evaluation time, build time, and statistics object size data...")

# Generating CSV file
csv_file_path = os.path.join(checkpoint_dir, 'benchmark_times.csv')

# Preparing CSV data
csv_data = []
for benchmark_name, times in all_benchmarks.items():
    build_time = times['BuildTime']
    eval_time = times['EvaluationTime']
    stats_size = times['StatisticsSize']
    parse_time = times['ParseTime']
    csv_data.append({
        'Benchmark': benchmark_name,
        'BuildTime': build_time if build_time is not None else '',
        'StatisticsSize': stats_size if stats_size is not None else '',
        'EvaluationTime': eval_time if eval_time is not None else '',
        'ParseTime': parse_time if parse_time is not None else ''
    })

# Writing CSV file
with open(csv_file_path, 'w', newline='', encoding='utf-8') as f:
    fieldnames = ['Benchmark', 'BuildTime', 'StatisticsSize', 'EvaluationTime', 'ParseTime']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(csv_data)

print(f"\nData saved to: {csv_file_path}")
print("\nCSV content preview:")
print(f"{'Benchmark':<20} {'BuildTime':<15} {'StatisticsSize':<18} {'EvaluationTime':<15} {'ParseTime':<12}")
print("-" * 85)
for row in csv_data:
    build_time = str(row['BuildTime']) if row['BuildTime'] else 'N/A'
    stats_size = str(row['StatisticsSize']) if row['StatisticsSize'] else 'N/A'
    eval_time = str(row['EvaluationTime']) if row['EvaluationTime'] else 'N/A'
    parse_t = str(row.get('ParseTime', '')) if row.get('ParseTime', '') else 'N/A'
    print(f"{row['Benchmark']:<20} {build_time:<15} {stats_size:<18} {eval_time:<15} {parse_t:<12}")



Data saved to: /home/liwei/starce-final/StarCE/experiment/checkpoint/SafeBound/benchmark_times.csv

CSV content preview:
Benchmark            BuildTime       StatisticsSize     EvaluationTime  ParseTime   
-------------------------------------------------------------------------------------
Stats                21.998126       1268042.0          2.6882646083831787 0.40495800971984863
JOBLight             242.268736      362156.0           0.36675047874450684 3.891819477081299
JOBLightRanges       391.001837      608152.0           10.708675146102905 85.28596520423889
JOBM                 2531.138521     3581644.0          80.82736206054688 121.196053981781
StatsJoin            20.81978        1268042.0          0.00033164024353027344 0.07736611366271973
